In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from collections import defaultdict

np.random.seed(42)

In [11]:
def solve_glft(Q, gamma, sigma, k, A, T, n_time_points=201):
    n_states = 2*Q + 1
    q_values = np.arange(-Q, Q+1)

    alpha = 0.5 * k * gamma * sigma**2
    eta = A * (1 + gamma/k) ** (-(1 + k/gamma))

    M = np.zeros((n_states, n_states))
    for idx, q in enumerate(q_values):
        M[idx, idx] = alpha * q**2
        if idx+1 < n_states:
            M[idx, idx+1] = -eta
        if idx-1 >= 0:
            M[idx, idx-1] = -eta

    tau_grid = np.linspace(0, T, n_time_points)
    dtau = tau_grid[1] - tau_grid[0]

    # Single SMALL-step matrix exponential -- well-conditioned, computed once
    step_matrix = expm(M * dtau)

    V_tau = np.zeros((n_time_points, n_states))
    v = np.ones(n_states)
    V_tau[0] = v

    for i in range(1, n_time_points):
        v = step_matrix @ v
        v = v / np.max(v)          # renormalize: keeps magnitudes safe, ratios unchanged
        V_tau[i] = v

    V = V_tau[::-1]
    t_grid = T - tau_grid[::-1]

    return t_grid, V, q_values, alpha, eta


def glft_quotes(q, t_idx, V, q_values, k, gamma):
    Q = q_values.max()
    idx = q + Q
    const = (1/gamma) * np.log(1 + gamma/k)

    v_q = max(V[t_idx, idx], 1e-300)   # floor to avoid log(0) / log(negative)

    delta_b = None
    if q < Q:
        v_qp1 = max(V[t_idx, idx+1], 1e-300)
        delta_b = (1/k) * np.log(v_q / v_qp1) + const

    delta_a = None
    if q > -Q:
        v_qm1 = max(V[t_idx, idx-1], 1e-300)
        delta_a = (1/k) * np.log(v_q / v_qm1) + const

    return delta_b, delta_a

In [12]:
class LimitOrderBook:
    def __init__(self, tick_size=0.01, initial_mid=100.0):
        self.tick_size = tick_size
        self.bids = defaultdict(list)
        self.asks = defaultdict(list)
        self.orders = {}
        self._id_counter = 0
        self.last_trade_price = initial_mid
        self.trade_log = []

    def _new_id(self):
        self._id_counter += 1
        return self._id_counter

    def best_bid(self):
        return max(self.bids) if self.bids else None

    def best_ask(self):
        return min(self.asks) if self.asks else None

    def mid_price(self):
        bb, ba = self.best_bid(), self.best_ask()
        if bb is not None and ba is not None:
            return (bb + ba) / 2
        return self.last_trade_price

    def add_limit_order(self, side, price, qty, owner):
        price = round(price / self.tick_size) * self.tick_size
        oid = self._new_id()
        book = self.bids if side == 'buy' else self.asks
        book[price].append([oid, qty, owner])
        self.orders[oid] = (side, price, owner)
        return oid

    def cancel_order(self, oid):
        if oid not in self.orders:
            return
        side, price, owner = self.orders.pop(oid)
        book = self.bids if side == 'buy' else self.asks
        book[price] = [o for o in book[price] if o[0] != oid]
        if not book[price]:
            del book[price]

    def market_order(self, side, qty, t, aggressor='noise'):
        fills = []
        book = self.asks if side == 'buy' else self.bids
        remaining = qty
        price_levels = sorted(book.keys()) if side == 'buy' else sorted(book.keys(), reverse=True)

        for price in price_levels:
            if remaining <= 0:
                break
            queue = book[price]
            while queue and remaining > 0:
                oid, oqty, owner = queue[0]
                traded = min(oqty, remaining)
                fills.append((owner, price, traded, side))
                self.trade_log.append((t, price, traded, aggressor))
                self.last_trade_price = price
                remaining -= traded
                oqty -= traded
                if oqty == 0:
                    queue.pop(0)
                    del self.orders[oid]
                else:
                    queue[0][1] = oqty
            if not queue:
                del book[price]

        return fills, remaining

    def depth_snapshot(self, n_levels=5):
        bid_prices = sorted(self.bids, reverse=True)[:n_levels]
        ask_prices = sorted(self.asks)[:n_levels]
        bid_depth = [(p, sum(o[1] for o in self.bids[p])) for p in bid_prices]
        ask_depth = [(p, sum(o[1] for o in self.asks[p])) for p in ask_prices]
        return bid_depth, ask_depth

In [13]:
class GLFTMarketMaker:
    def __init__(self, name, Q_max, gamma, sigma, k, A, T, quote_size=1):
        self.name = name
        self.Q_max = Q_max
        self.gamma, self.sigma, self.k, self.A, self.T = gamma, sigma, k, A, T
        self.quote_size = quote_size

        self.t_grid, self.V, self.q_values, _, _ = solve_glft(Q_max, gamma, sigma, k, A, T)

        self.inventory = 0
        self.cash = 0.0
        self.active_orders = []
        self.history = {'t': [], 'inventory': [], 'cash': [], 'wealth': []}

    def _time_index(self, t):
        return min(np.searchsorted(self.t_grid, t), len(self.t_grid) - 1)

    def update_quotes(self, lob, t, fair_value):
        for oid in self.active_orders:
            lob.cancel_order(oid)
        self.active_orders = []

        t_idx = self._time_index(t)
        q = int(np.clip(self.inventory, -self.Q_max, self.Q_max))
        delta_b, delta_a = glft_quotes(q, t_idx, self.V, self.q_values, self.k, self.gamma)

        if delta_b is not None:
            bid_price = fair_value - delta_b
            oid = lob.add_limit_order('buy', bid_price, self.quote_size, self.name)
            self.active_orders.append(oid)
        if delta_a is not None:
            ask_price = fair_value + delta_a
            oid = lob.add_limit_order('sell', ask_price, self.quote_size, self.name)
            self.active_orders.append(oid)

    def process_fills(self, fills):
        # FIX: 'side' is the AGGRESSOR's side. An aggressor 'buy' consumes our ASK
        # (we sold), an aggressor 'sell' consumes our BID (we bought).
        for owner, price, qty, side in fills:
            if owner != self.name:
                continue
            if side == 'buy':      # aggressor bought -> hit our ask -> we sold
                self.inventory -= qty
                self.cash += price * qty
            else:                  # aggressor sold -> hit our bid -> we bought
                self.inventory += qty
                self.cash -= price * qty

    def record(self, t, mid_price):
        self.history['t'].append(t)
        self.history['inventory'].append(self.inventory)
        self.history['cash'].append(self.cash)
        self.history['wealth'].append(self.cash + self.inventory * mid_price)


class NaiveMarketMaker:
    def __init__(self, name, half_spread, quote_size=1):
        self.name = name
        self.half_spread = half_spread
        self.quote_size = quote_size
        self.inventory = 0
        self.cash = 0.0
        self.active_orders = []
        self.history = {'t': [], 'inventory': [], 'cash': [], 'wealth': []}

    def update_quotes(self, lob, t, fair_value):
        for oid in self.active_orders:
            lob.cancel_order(oid)
        self.active_orders = []
        oid_b = lob.add_limit_order('buy', fair_value - self.half_spread, self.quote_size, self.name)
        oid_a = lob.add_limit_order('sell', fair_value + self.half_spread, self.quote_size, self.name)
        self.active_orders = [oid_b, oid_a]

    def process_fills(self, fills):
        for owner, price, qty, side in fills:
            if owner != self.name:
                continue
            if side == 'buy':
                self.inventory -= qty; self.cash += price * qty
            else:
                self.inventory += qty; self.cash -= price * qty

    def record(self, t, mid_price):
        self.history['t'].append(t)
        self.history['inventory'].append(self.inventory)
        self.history['cash'].append(self.cash)
        self.history['wealth'].append(self.cash + self.inventory * mid_price)

In [14]:
class NoiseTraders:
    def __init__(self, arrival_rate, size_choices=(1,2,3)):
        self.arrival_rate = arrival_rate
        self.size_choices = size_choices

    def generate_orders(self, dt):
        n_orders = np.random.poisson(self.arrival_rate * dt)
        orders = []
        for _ in range(n_orders):
            side = np.random.choice(['buy', 'sell'])
            qty = np.random.choice(self.size_choices)
            orders.append((side, qty))
        return orders


class InformedTraders:
    def __init__(self, arrival_rate, sensitivity=0.5, size_choices=(1,2)):
        self.arrival_rate = arrival_rate
        self.sensitivity = sensitivity
        self.size_choices = size_choices

    def generate_orders(self, dt, fundamental_value, mid_price):
        n_orders = np.random.poisson(self.arrival_rate * dt)
        gap = fundamental_value - mid_price
        prob_buy = 1 / (1 + np.exp(-self.sensitivity * gap))
        orders = []
        for _ in range(n_orders):
            side = 'buy' if np.random.rand() < prob_buy else 'sell'
            qty = np.random.choice(self.size_choices)
            orders.append((side, qty))
        return orders

In [15]:
def run_lob_simulation(market_makers, T=1.0, n_steps=500, sigma_fundamental=2.0,
                        noise_rate=80, informed_rate=15, tick_size=0.01, initial_mid=100.0):
    dt = T / n_steps
    lob = LimitOrderBook(tick_size=tick_size, initial_mid=initial_mid)

    noise_traders = NoiseTraders(arrival_rate=noise_rate)
    informed_traders = InformedTraders(arrival_rate=informed_rate)

    fundamental_value = initial_mid
    fundamental_path = [fundamental_value]
    mid_path = [initial_mid]
    time_path = [0.0]

    for step in range(n_steps):
        t = step * dt
        fundamental_value += sigma_fundamental * np.sqrt(dt) * np.random.randn()

        for mm in market_makers:
            mm.update_quotes(lob, t, fundamental_value)

        for side, qty in noise_traders.generate_orders(dt):
            fills, _ = lob.market_order(side, qty, t, aggressor='noise')
            for mm in market_makers:
                mm.process_fills(fills)

        current_mid = lob.mid_price()
        for side, qty in informed_traders.generate_orders(dt, fundamental_value, current_mid):
            fills, _ = lob.market_order(side, qty, t, aggressor='informed')
            for mm in market_makers:
                mm.process_fills(fills)

        mid_now = lob.mid_price()
        for mm in market_makers:
            mm.record(t, mid_now)

        fundamental_path.append(fundamental_value)
        mid_path.append(mid_now)
        time_path.append(t + dt)

    return lob, time_path, fundamental_path, mid_path

In [24]:
Q_max, gamma, sigma, k, A, T = 1000, 0.01, 2.0, 1.5, 140.0, 1.0

mm_glft  = GLFTMarketMaker('GLFT_MM', Q_max, gamma, sigma, k, A, T, quote_size=2)
mm_naive = NaiveMarketMaker('Naive_MM', half_spread=(1/gamma)*np.log(1+gamma/k), quote_size=2)

np.random.seed(7)
lob, time_path, fundamental_path, mid_path = run_lob_simulation(
    [mm_glft, mm_naive], T=T, n_steps=500,
    sigma_fundamental=sigma, noise_rate=80, informed_rate=15
)

print(f"GLFT MM  -> final inventory: {mm_glft.inventory}, final wealth: {mm_glft.history['wealth'][-1]:.2f}")
print(f"Naive MM -> final inventory: {mm_naive.inventory}, final wealth: {mm_naive.history['wealth'][-1]:.2f}")

GLFT MM  -> final inventory: -6, final wealth: 122.83
Naive MM -> final inventory: -5, final wealth: 25.69


In [25]:
max_glft_q = max(abs(x) for x in mm_glft.history['inventory'])
print(f"Max |inventory| reached by GLFT MM on the real order book: {max_glft_q} (bound is {Q_max})")

Max |inventory| reached by GLFT MM on the real order book: 12 (bound is 1000)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

axes[0].plot(time_path, fundamental_path, label='Fundamental value', color='black', alpha=0.6)
axes[0].plot(time_path, mid_path, label='LOB mid-price (emergent)', color='tab:blue')
axes[0].set_ylabel('Price'); axes[0].legend()
axes[0].set_title('Price Discovery: Fundamental vs Order-Book Mid-Price')

axes[1].plot(mm_glft.history['t'], mm_glft.history['inventory'], label='GLFT MM inventory')
axes[1].plot(mm_naive.history['t'], mm_naive.history['inventory'], label='Naive MM inventory')
axes[1].axhline(Q_max, color='red', linestyle=':', alpha=0.5)
axes[1].axhline(-Q_max, color='red', linestyle=':', alpha=0.5)
axes[1].set_ylabel('Inventory'); axes[1].legend()
axes[1].set_title('Inventory Control on the Real Order Book')

axes[2].plot(mm_glft.history['t'], mm_glft.history['wealth'], label='GLFT MM wealth')
axes[2].plot(mm_naive.history['t'], mm_naive.history['wealth'], label='Naive MM wealth')
axes[2].set_ylabel('Wealth'); axes[2].set_xlabel('Time'); axes[2].legend()
axes[2].set_title('Mark-to-Market Wealth')

plt.tight_layout()
plt.show()

In [ ]:
n_sims = 200
results_glft, results_naive, max_q_glft = [], [], []

for i in range(n_sims):
    mm_glft_i  = GLFTMarketMaker('GLFT_MM', Q_max, gamma, sigma, k, A, T, quote_size=2)
    mm_naive_i = NaiveMarketMaker('Naive_MM', half_spread=(1/gamma)*np.log(1+gamma/k), quote_size=2)
    np.random.seed(i)
    run_lob_simulation([mm_glft_i, mm_naive_i], T=T, n_steps=300,
                        sigma_fundamental=sigma, noise_rate=80, informed_rate=15)
    results_glft.append(mm_glft_i.history['wealth'][-1])
    results_naive.append(mm_naive_i.history['wealth'][-1])
    max_q_glft.append(max(abs(x) for x in mm_glft_i.history['inventory']))

results_glft = np.array(results_glft)
results_naive = np.array(results_naive)

print(f"GLFT MM  -> mean PnL: {results_glft.mean():.3f}, std: {results_glft.std():.3f}, "
      f"Sharpe-like: {results_glft.mean()/results_glft.std():.3f}")
print(f"Naive MM -> mean PnL: {results_naive.mean():.3f}, std: {results_naive.std():.3f}, "
      f"Sharpe-like: {results_naive.mean()/results_naive.std():.3f}")
print(f"Max |inventory| ever reached across all {n_sims} sims: {max(max_q_glft)} (bound is {Q_max})")

plt.figure(figsize=(10,5))
plt.hist(results_glft, bins=30, alpha=0.6, label='GLFT MM')
plt.hist(results_naive, bins=30, alpha=0.6, label='Naive MM')
plt.xlabel('Final PnL'); plt.ylabel('Frequency')
plt.title(f'PnL Distribution — Real Order Book, {n_sims} runs')
plt.legend()
plt.show()